# So sánh Feature Selection: Boruta vs Lasso — Superconductivity

Tương tự `test_housing_lassovsboruta.ipynb` nhưng cho bộ **Superconductivity**
(`superconductivity_processed.csv`, target = `critical_temp`, **81 feature số**).
So sánh R²: **baseline GBM (full)** vs **Boruta (RF)** vs **Lasso**.

Đây là kịch bản **"đa cộng tuyến nặng"** (74 cặp |r|>0,9) → selection sẽ kém ổn định trong các nhóm feature trùng lặp.

> ⏱️ Lưu ý: 21k dòng × 81 feature → **Boruta chạy khá lâu** (vài phút) trên Colab. LassoCV cũng tốn thời gian; có thể giảm `cv=3` nếu cần nhanh.

In [1]:
!pip install Boruta -q

import pandas as pd
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from boruta import BorutaPy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 2.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/AIO-Conquer02')
os.getcwd()

Mounted at /content/drive


'/content/drive/MyDrive/AIO-Conquer02'

In [3]:
import pandas as pd
df = pd.read_csv('superconductivity_processed.csv')
print('Shape:', df.shape)
df.info(verbose=False)

Shape: (21197, 82)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21197 entries, 0 to 21196
Columns: 82 entries, number_of_elements to critical_temp
dtypes: float64(79), int64(3)
memory usage: 13.3 MB


### Chuẩn bị dữ liệu

Bộ này **đã processed thành toàn số** (không có cột chữ → **không cần encode**). Chỉ tách X/y và chia train/test.
Target `critical_temp` giữ **nguyên gốc**.

In [4]:
target_colum = 'critical_temp'
X = df.drop(columns=[target_colum], errors='ignore')
y = df[target_colum]

# Chia Train/Test trước để tránh data leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Số feature:', X.shape[1])

Số feature: 81


## Baseline — Gradient Boosting với toàn bộ feature

In [5]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

gbr = GradientBoostingRegressor(max_depth=5, random_state=42)
gbr.fit(X_train, y_train)
preds = gbr.predict(X_test)
r2_score_all = round(r2_score(y_test, preds), 3)
print(f"Chỉ số R2-score khi sử dụng tất cả feature: {r2_score_all}")

Chỉ số R2-score khi sử dụng tất cả feature: 0.904


## Boruta (lõi Random Forest)

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from boruta import BorutaPy

# --- Bước 1: Tên feature ban đầu ---
feature_names = X.columns.tolist()

# --- Bước 2: Chuyển sang Numpy để tránh lỗi Index ---
X_train_np = X_train.values if hasattr(X_train, 'columns') else X_train
X_test_np = X_test.values if hasattr(X_test, 'columns') else X_test
y_train_np = y_train.values if hasattr(y_train, 'values') else y_train
y_test_np = y_test.values if hasattr(y_test, 'values') else y_test

# --- Bước 3: Mô hình lõi ---
rf = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)

# --- Bước 4: Cấu hình & chạy Boruta ---
boruta_selector = BorutaPy(
    estimator=rf,
    n_estimators='auto',
    verbose=0,
    random_state=42,
    max_iter=100
)
boruta_selector.fit(X_train_np, y_train_np)

# --- Bước 5: Lọc feature được chọn ---
X_train_filtered = boruta_selector.transform(X_train_np)
X_test_filtered = boruta_selector.transform(X_test_np)

# --- Bước 6: Huấn luyện lại trên feature đã lọc ---
rf_final = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
rf_final.fit(X_train_filtered, y_train_np)
preds_boruta = rf_final.predict(X_test_filtered)
r2_score_boruta = round(r2_score(y_test_np, preds_boruta), 3)

confirmed_features = [feature_names[i] for i, c in enumerate(boruta_selector.support_) if c]

print("="*60)
print(f"Chỉ số R2-score khi sử dụng Boruta: {r2_score_boruta}")
print(f"Số lượng tính năng được giữ lại: {len(confirmed_features)} / {len(feature_names)}")
print(f"Các tính năng được chọn: {confirmed_features}")
print("="*60)

Chỉ số R2-score khi sử dụng Boruta: 0.807
Số lượng tính năng được giữ lại: 52 / 81
Các tính năng được chọn: ['wtd_mean_atomic_mass', 'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass', 'wtd_entropy_atomic_mass', 'range_atomic_mass', 'wtd_range_atomic_mass', 'std_atomic_mass', 'wtd_std_atomic_mass', 'wtd_mean_fie', 'gmean_fie', 'wtd_entropy_fie', 'range_fie', 'wtd_range_fie', 'wtd_mean_atomic_radius', 'range_atomic_radius', 'wtd_range_atomic_radius', 'std_atomic_radius', 'wtd_std_atomic_radius', 'mean_Density', 'gmean_Density', 'wtd_gmean_Density', 'entropy_Density', 'wtd_entropy_Density', 'std_Density', 'mean_ElectronAffinity', 'gmean_ElectronAffinity', 'wtd_gmean_ElectronAffinity', 'entropy_ElectronAffinity', 'wtd_entropy_ElectronAffinity', 'range_ElectronAffinity', 'wtd_range_ElectronAffinity', 'wtd_std_ElectronAffinity', 'mean_FusionHeat', 'entropy_FusionHeat', 'wtd_entropy_FusionHeat', 'wtd_std_FusionHeat', 'wtd_mean_ThermalConductivity', 'gmean_ThermalConductivity

## Lasso (LassoCV, có chuẩn hoá)

In [7]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import numpy as np

# --- Bước 1: Chuẩn hoá (Lasso cần cùng thang đo để phạt công bằng) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Bước 2: LassoCV tự tìm alpha tối ưu ---
lasso = LassoCV(
    alphas=np.logspace(-4, 1, 100),
    cv=5,
    max_iter=10000,
    tol=0.01,
    random_state=42,
    n_jobs=-1
)
lasso.fit(X_train_scaled, y_train)

# --- Bước 3: Dự đoán & R2 ---
preds_lasso = lasso.predict(X_test_scaled)
r2_score_lasso = round(r2_score(y_test, preds_lasso), 3)

# --- Bước 4: Feature bị Lasso loại (hệ số = 0) ---
feature_names = X.columns if hasattr(X, 'columns') else [f"Feature_{i}" for i in range(X.shape[1])]
coefficients = lasso.coef_
kept_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef != 0]
dropped_features = [feature_names[i] for i, coef in enumerate(coefficients) if coef == 0]

# --- Bước 5: So sánh với baseline ---
print("="*60)
print(f"[-] R2-score Baseline (Gradient Boosting): {r2_score_all}")
print(f"[+] R2-score khi sử dụng Lasso: {r2_score_lasso}")
print(f" Alpha tối ưu: {round(lasso.alpha_, 5)}")
print("="*60)
print(f" Số feature bị Lasso loại (hệ số = 0): {len(dropped_features)} / {len(feature_names)}")
if len(dropped_features) > 0:
    print(f"   Bị loại: {dropped_features}")
print(f"\n Số feature giữ lại: {len(kept_features)}")
print("="*60)

[-] R2-score Baseline (Gradient Boosting): 0.904
[+] R2-score khi sử dụng Lasso: 0.735
 Alpha tối ưu: 0.0001
 Số feature bị Lasso loại (hệ số = 0): 0 / 81

 Số feature giữ lại: 81


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.427e+06, tolerance: 1.991e+05
  model = cd_fast.enet_coordinate_descent(


## Kết quả & nhận xét

Số liệu **GBR/Lasso đã kiểm chứng local** (deterministic vì cố định `random_state=42` → chạy Colab ra y hệt). Boruta chạy trên Colab.

| Phương pháp | R² (test) | Feature giữ lại |
|---|---|---|
| **Baseline — Gradient Boosting (full)** | **0,904** | 81/81 |
| **Lasso (LassoCV)** | 0,735 | 81/81 (**bỏ 0**, alpha≈0,0001) |
| **Boruta (RF)** | 0.807 | 52 / 81 |

**🔍 Điểm mấu chốt:** ngược với Communities — ở đây **GBM (0,904) ≫ Lasso (0,735)** và **Lasso không loại được feature nào** (alpha tối ưu nhỏ xíu ≈0,0001, giữ cả 81 cột).

Hai nguyên nhân:
1. **Đa cộng tuyến nặng** (74 cặp |r|>0,9): mọi feature đều "có ích" nên Lasso không dám ép về 0 — selection **không cắt được gì**, giống bài học bộ housing.
2. **Phi tuyến mạnh**: chênh lệch GBM≫Lasso cho thấy quan hệ `critical_temp` ↔ feature phi tuyến rõ → model tuyến tính (kể cả sau selection) thua hẳn cây.

→ Đối chiếu 2 bộ cho bức tranh đầy đủ: **Communities = "nhiều nhiễu" → selection giúp ích**; **Superconductivity = "trùng lặp + phi tuyến" → selection bất lực, nên dùng model phi tuyến**. Nhớ `StandardScaler` trước Lasso vì feature ở thang khác nhau.